# 🌿 ArvyaX Emotion Intelligence Pipeline
> **ArvyaX × RevoltronX ML Internship Assignment**

Run each cell top to bottom. Everything runs locally inside Colab (no external APIs).

---
**Sections:**
1. Install dependencies
2. Generate data
3. Feature engineering
4. Train models (State + Intensity)
5. Decision engine
6. Uncertainty modeling
7. Ablation study
8. Error analysis
9. Robustness tests
10. Save & download outputs

## ⚙️ Cell 1 — Install Dependencies

In [ ]:
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn -q
print('✅ All packages installed')

## 📦 Cell 2 — Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import random, pickle, os
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, accuracy_score,
    mean_absolute_error, mean_squared_error, confusion_matrix
)
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.calibration import CalibratedClassifierCV
import xgboost as xgb

random.seed(42)
np.random.seed(42)
print('✅ Imports done')

## 🗄️ Cell 3 — Generate Dataset

In [ ]:
# ── Schema matches ArvyaX dataset exactly ──
EMOTIONAL_STATES = ['calm','anxious','focused','tired','happy',
                    'sad','restless','content','overwhelmed','hopeful']
AMBIENCE_TYPES   = ['forest','ocean','rain','mountain','café']
TIME_OF_DAY      = ['morning','afternoon','evening','night']
FACE_HINTS       = ['neutral','happy','sad','tense','tired','surprised','unknown']
PREV_MOODS       = ['good','bad','neutral','excellent','terrible']
REFLECTION_QUALITY = ['high','medium','low']

TEXT_TEMPLATES = {
    'calm':        ['I feel really settled today. The session helped me slow down.',
                    'Everything feels light and easy. I am at peace with where I am.',
                    'Breathing was smooth. Mind is quiet. Ready to move through the day.',
                    'The rain sounds calmed me completely. I feel still inside.',
                    'Nothing feels urgent right now. Just present.', 'ok', 'fine.'],
    'anxious':     ['I cannot stop thinking about everything I need to do. Heart is racing.',
                    'My mind keeps jumping. I tried to focus but it is hard.',
                    'Feel like something bad is about to happen but I do not know what.',
                    'So many things on my plate. Cannot settle.',
                    'Restless. Cannot stop fidgeting. The session helped a little but not much.',
                    'worried about tomorrow', 'anxious, nervous, not sure why'],
    'focused':     ['Crystal clear today. The forest sounds helped me zone in completely.',
                    'Ready to work. Mind feels sharp. Very productive headspace.',
                    'I have a clear plan. The session helped me prioritize.',
                    'Laser focused. I know exactly what needs to get done.',
                    'Good energy, clear head. Lets go.', 'focused'],
    'tired':       ['Exhausted. Could not sleep last night. Dragging myself through the morning.',
                    'The session was nice but I still feel heavy. Need rest badly.',
                    'Low energy. Everything feels slow and foggy.',
                    'I feel drained even after the session. Sleep deprived for sure.',
                    'My body is tired. Mind is foggy. Hard to think clearly.',
                    'so tired', 'just need sleep honestly'],
    'happy':       ['Feeling great! The session was wonderful and I feel so alive.',
                    'So much joy today. Grateful for this moment.',
                    'Everything feels right. Light, energetic, and happy.',
                    'The ocean sounds put me in a wonderful mood. Cannot stop smiling.',
                    'Life is good. I feel optimistic and ready for anything.', 'amazing session!'],
    'sad':         ['Feeling low. Hard to explain why. Just sad.',
                    'The session helped a little, but the heaviness is still there.',
                    'I miss someone. The rain sounds made it worse somehow.',
                    'Nothing feels exciting. Everything feels grey.',
                    'Cried a little during the session. Needed that release.',
                    'just sad', 'not good'],
    'restless':    ['Cannot sit still. Lots of energy but it is scattered.',
                    'My thoughts are all over the place. I need direction.',
                    'Feel like I need to move or do something. Cannot relax.',
                    'Agitated. The café sounds did not help today.',
                    'Something is off. I feel unsettled but not anxious exactly.',
                    'restless and a bit scattered'],
    'content':     ['I feel satisfied. Not euphoric, just really okay.',
                    'Good session. I feel grounded and content with where I am.',
                    'Everything is fine. Calm but engaged. Ready for a normal day.',
                    'Peaceful and okay. Nothing special, just good.',
                    'Content. Ready. Settled.', 'it was alright'],
    'overwhelmed': ['Too much. Too many thoughts. Cannot organize any of them.',
                    'I broke down a little during the session. Everything feels like too much.',
                    'Pressure from all sides. I do not know where to start.',
                    'My head is spinning. I need to stop and breathe.',
                    'Overwhelmed is an understatement. I am drowning in tasks and feelings.',
                    'too much everything', 'cannot cope right now'],
    'hopeful':     ['I feel like things are turning around. Today could be a good day.',
                    'The mountain session gave me perspective. I see light ahead.',
                    'Optimistic. I believe in what I am working toward.',
                    'Something shifted in me today. I feel hopeful and ready.',
                    'The future does not scare me right now. I feel ready.',
                    'feeling better than yesterday'],
}

STATE_CONTEXT = {
    'calm':        dict(sleep=(6.5,8.5), energy=(3,5), stress=(1,3)),
    'anxious':     dict(sleep=(4.5,6.5), energy=(2,4), stress=(3,5)),
    'focused':     dict(sleep=(6.5,8.5), energy=(3,5), stress=(2,4)),
    'tired':       dict(sleep=(3.5,6.0), energy=(1,3), stress=(2,4)),
    'happy':       dict(sleep=(6.0,9.0), energy=(3,5), stress=(1,3)),
    'sad':         dict(sleep=(4.5,7.5), energy=(1,3), stress=(2,5)),
    'restless':    dict(sleep=(5.0,7.0), energy=(3,5), stress=(2,4)),
    'content':     dict(sleep=(6.5,8.5), energy=(3,5), stress=(1,3)),
    'overwhelmed': dict(sleep=(4.0,6.5), energy=(1,3), stress=(4,5)),
    'hopeful':     dict(sleep=(6.0,8.0), energy=(2,4), stress=(1,3)),
}
STATE_INTENSITY = {
    'calm':(1,3),'anxious':(3,5),'focused':(3,5),'tired':(2,4),
    'happy':(3,5),'sad':(2,4),'restless':(2,4),'content':(1,3),
    'overwhelmed':(3,5),'hopeful':(2,4),
}

def generate_row(row_id, emotional_state, add_noise=True):
    ctx = STATE_CONTEXT[emotional_state]
    label = emotional_state
    if add_noise and random.random() < 0.10:
        label = random.choice(EMOTIONAL_STATES)
    sleep  = round(np.clip(np.random.normal((ctx['sleep'][0]+ctx['sleep'][1])/2, 1.0), 2.5, 10.0), 1)
    energy = int(np.clip(np.random.normal((ctx['energy'][0]+ctx['energy'][1])/2, 0.8), 1, 5))
    stress = int(np.clip(np.random.normal((ctx['stress'][0]+ctx['stress'][1])/2, 0.8), 1, 5))
    intensity = random.randint(*STATE_INTENSITY[emotional_state])
    if add_noise and random.random() < 0.08:
        stress = random.randint(1,5); energy = random.randint(1,5)
    if add_noise and random.random() < 0.05: sleep = np.nan
    if add_noise and random.random() < 0.05: energy = np.nan
    if add_noise and random.random() < 0.03: stress = np.nan
    return {
        'id': row_id,
        'journal_text': random.choice(TEXT_TEMPLATES[emotional_state]),
        'ambience_type': random.choice(AMBIENCE_TYPES),
        'duration_min': random.randint(5,45),
        'sleep_hours': sleep, 'energy_level': energy, 'stress_level': stress,
        'time_of_day': random.choice(TIME_OF_DAY),
        'previous_day_mood': random.choice(PREV_MOODS),
        'face_emotion_hint': random.choice(FACE_HINTS),
        'reflection_quality': random.choice(REFLECTION_QUALITY),
        'emotional_state': label, 'intensity': intensity,
    }

def generate_dataset(n, start_id=1, is_train=True):
    rows, row_id = [], start_id
    per_class = n // len(EMOTIONAL_STATES)
    for state in EMOTIONAL_STATES:
        for _ in range(per_class):
            rows.append(generate_row(row_id, state, add_noise=is_train))
            row_id += 1
    while len(rows) < n:
        rows.append(generate_row(row_id, random.choice(EMOTIONAL_STATES), add_noise=is_train))
        row_id += 1
    random.shuffle(rows)
    return pd.DataFrame(rows)

train_df = generate_dataset(500, start_id=1, is_train=True)
test_df_full = generate_dataset(100, start_id=501, is_train=False)
test_df = test_df_full.drop(columns=['emotional_state','intensity'])

print(f'✅ Train: {train_df.shape}, Test: {test_df.shape}')
print(f'Class distribution:')
print(train_df['emotional_state'].value_counts())
train_df.head(3)

## 🔧 Cell 4 — Feature Engineering (Part 5)

In [ ]:
class FeatureEngineer:
    def __init__(self, max_features=500):
        self.tfidf = TfidfVectorizer(max_features=max_features, ngram_range=(1,2),
                                      sublinear_tf=True, stop_words='english', min_df=1)
        self.scaler  = StandardScaler()
        self.imputer = SimpleImputer(strategy='median')
        self.meta_cols = ['duration_min','sleep_hours','energy_level','stress_level',
                          'time_of_day_enc','previous_day_mood_enc',
                          'face_emotion_hint_enc','reflection_quality_enc','ambience_type_enc']
        self.ordinal_maps = {
            'time_of_day':       {'morning':0,'afternoon':1,'evening':2,'night':3},
            'previous_day_mood': {'terrible':0,'bad':1,'neutral':2,'good':3,'excellent':4},
            'reflection_quality':{'low':0,'medium':1,'high':2},
            'face_emotion_hint': {'sad':0,'tired':1,'tense':2,'neutral':3,'surprised':4,'happy':5,'unknown':3},
            'ambience_type':     {'rain':0,'café':1,'forest':2,'ocean':3,'mountain':4},
        }
        self.fitted = False

    def _encode_ordinals(self, df):
        d = df.copy()
        for col, mapping in self.ordinal_maps.items():
            d[col+'_enc'] = d[col].map(mapping).fillna(2)
        return d

    def _text_extras(self, texts):
        lengths     = texts.str.len().fillna(0).values.reshape(-1,1)
        word_counts = texts.str.split().str.len().fillna(0).values.reshape(-1,1)
        is_short    = (word_counts < 5).astype(float)
        return np.hstack([lengths, word_counts, is_short])

    def fit_transform(self, df):
        df = self._encode_ordinals(df)
        texts = df['journal_text'].fillna('').astype(str)
        tfidf = self.tfidf.fit_transform(texts).toarray()
        extra = self._text_extras(texts)
        meta  = self.imputer.fit_transform(df[self.meta_cols].values.astype(float))
        meta  = self.scaler.fit_transform(meta)
        self.fitted = True
        return np.hstack([tfidf, extra, meta])

    def transform(self, df):
        df = self._encode_ordinals(df)
        texts = df['journal_text'].fillna('').astype(str)
        tfidf = self.tfidf.transform(texts).toarray()
        extra = self._text_extras(texts)
        meta  = self.imputer.transform(df[self.meta_cols].values.astype(float))
        meta  = self.scaler.transform(meta)
        return np.hstack([tfidf, extra, meta])

    def text_only_fit_transform(self, df):
        texts = df['journal_text'].fillna('').astype(str)
        tfidf = self.tfidf.fit_transform(texts).toarray()
        self.fitted = True
        return np.hstack([tfidf, self._text_extras(texts)])

    def text_only_transform(self, df):
        texts = df['journal_text'].fillna('').astype(str)
        return np.hstack([self.tfidf.transform(texts).toarray(), self._text_extras(texts)])

fe = FeatureEngineer()
X_train = fe.fit_transform(train_df)
X_test  = fe.transform(test_df_full)
print(f'✅ Feature matrix — Train: {X_train.shape}, Test: {X_test.shape}')

## 🧠 Cell 5 — Train Emotional State Model (Part 1)

In [ ]:
le = LabelEncoder()
y_train_state = le.fit_transform(train_df['emotional_state'])
y_test_state  = le.transform(test_df_full['emotional_state'])

# XGBoost + Calibration for reliable confidence scores
base_clf = xgb.XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    use_label_encoder=False, eval_metric='mlogloss', random_state=42
)
state_model = CalibratedClassifierCV(base_clf, method='sigmoid', cv=3)
state_model.fit(X_train, y_train_state)

# Cross-validation
cv_scores = cross_val_score(
    xgb.XGBClassifier(n_estimators=100, use_label_encoder=False,
                      eval_metric='mlogloss', random_state=42),
    X_train, y_train_state, cv=StratifiedKFold(5), scoring='accuracy'
)
print(f'📊 Cross-val Accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

# Test evaluation
y_pred_state = state_model.predict(X_test)
y_pred_labels = le.inverse_transform(y_pred_state)
print(f'\n🎯 Test Accuracy: {accuracy_score(y_test_state, y_pred_state):.3f}')
print('\n📋 Classification Report:')
print(classification_report(test_df_full['emotional_state'], y_pred_labels))

## 📈 Cell 6 — Confusion Matrix Visualization

In [ ]:
cm = confusion_matrix(test_df_full['emotional_state'], y_pred_labels, labels=EMOTIONAL_STATES)
plt.figure(figsize=(12, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=EMOTIONAL_STATES, yticklabels=EMOTIONAL_STATES)
plt.title('🌿 ArvyaX — Emotional State Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('True State'); plt.xlabel('Predicted State')
plt.tight_layout(); plt.show()

## 📊 Cell 7 — Intensity Model (Part 2)

In [ ]:
# Treated as REGRESSION (ordinal scale 1-5)
# Regression respects ordinality — being off by 1 is better than off by 3
y_train_int = train_df['intensity'].values.astype(float)
y_test_int  = test_df_full['intensity'].values.astype(float)

intensity_model = xgb.XGBRegressor(
    n_estimators=150, max_depth=5, learning_rate=0.1,
    subsample=0.8, random_state=42
)
intensity_model.fit(X_train, y_train_int)

int_pred_raw = intensity_model.predict(X_test)
int_pred = np.clip(np.round(int_pred_raw), 1, 5).astype(int)

mae  = mean_absolute_error(y_test_int, int_pred)
rmse = np.sqrt(mean_squared_error(y_test_int, int_pred))
print(f'📈 Intensity — MAE: {mae:.3f} | RMSE: {rmse:.3f}')
print(f'   (Off by less than {mae:.1f} intensity levels on average)')

# Distribution plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
pd.Series(y_test_int).value_counts().sort_index().plot(kind='bar', ax=axes[0],
    color='steelblue', title='True Intensity Distribution')
pd.Series(int_pred).value_counts().sort_index().plot(kind='bar', ax=axes[1],
    color='tomato', title='Predicted Intensity Distribution')
plt.tight_layout(); plt.show()

## 🧭 Cell 8 — Decision Engine (Part 3)

In [ ]:
class DecisionEngine:
    WHAT_MAP = {
        ('negative_high','high_intensity','low_energy'):  'rest',
        ('negative_high','high_intensity','high_energy'): 'box_breathing',
        ('negative_high','low_intensity', 'low_energy'):  'rest',
        ('negative_high','low_intensity', 'high_energy'): 'grounding',
        ('negative_low', 'high_intensity','low_energy'):  'rest',
        ('negative_low', 'high_intensity','high_energy'): 'journaling',
        ('negative_low', 'low_intensity', 'low_energy'):  'sound_therapy',
        ('negative_low', 'low_intensity', 'high_energy'): 'light_planning',
        ('positive_high','high_intensity','low_energy'):  'movement',
        ('positive_high','high_intensity','high_energy'): 'deep_work',
        ('positive_high','low_intensity', 'low_energy'):  'yoga',
        ('positive_high','low_intensity', 'high_energy'): 'deep_work',
        ('positive_low', 'high_intensity','low_energy'):  'pause',
        ('positive_low', 'high_intensity','high_energy'): 'light_planning',
        ('positive_low', 'low_intensity', 'low_energy'):  'rest',
        ('positive_low', 'low_intensity', 'high_energy'): 'light_planning',
        ('neutral',      'high_intensity','low_energy'):  'grounding',
        ('neutral',      'high_intensity','high_energy'): 'journaling',
        ('neutral',      'low_intensity', 'low_energy'):  'rest',
        ('neutral',      'low_intensity', 'high_energy'): 'light_planning',
    }
    NEG_HIGH = {'anxious','overwhelmed'}
    NEG_LOW  = {'sad','tired'}
    POS_HIGH = {'happy','focused'}
    POS_LOW  = {'calm','content','hopeful'}

    def _sg(self, s):
        if s in self.NEG_HIGH: return 'negative_high'
        if s in self.NEG_LOW:  return 'negative_low'
        if s in self.POS_HIGH: return 'positive_high'
        if s in self.POS_LOW:  return 'positive_low'
        return 'neutral'

    def decide_what(self, state, intensity, energy, stress):
        sg = self._sg(state)
        ig = 'high_intensity' if intensity >= 3 else 'low_intensity'
        eg = 'high_energy' if (not pd.isna(energy) and float(energy) >= 3) else 'low_energy'
        action = self.WHAT_MAP.get((sg, ig, eg), 'pause')
        if not pd.isna(stress) and float(stress) >= 4 and action == 'deep_work':
            action = 'box_breathing'
        return action

    def decide_when(self, state, intensity, energy, stress, time_of_day):
        if state in self.NEG_HIGH and intensity >= 4: return 'now'
        if time_of_day in ('evening','night'):
            return 'now' if state in self.NEG_HIGH and intensity >= 3 else 'tonight'
        if time_of_day == 'morning':
            return 'later_today' if state in ('tired','sad') else 'within_15_min'
        if time_of_day == 'afternoon':
            return 'later_today' if (state == 'tired' or (not pd.isna(energy) and float(energy) <= 2)) else 'within_15_min'
        return 'later_today'

    def message(self, state, intensity, what, when):
        templates = {
            ('anxious','box_breathing'):   'You seem quite anxious. A short breathing exercise will help settle your nervous system.',
            ('overwhelmed','box_breathing'):'That is a lot. Let us breathe through this. Box breathing can reset your mind in minutes.',
            ('tired','rest'):              'Your body is telling you something. Listen to it — rest is productive too.',
            ('sad','journaling'):          'It is okay to feel what you feel. Writing it down gives your emotions a place to land.',
            ('focused','deep_work'):       'You are in a great headspace. Use this clarity — perfect time for deep work.',
            ('happy','deep_work'):         'Great energy today! Channel it into something meaningful.',
            ('calm','light_planning'):     'You are grounded and clear. Gentle planning now could set up a smooth day.',
            ('restless','movement'):       'That restless energy needs somewhere to go. Movement will channel it productively.',
            ('hopeful','light_planning'):  'Great mindset today. Use that optimism to lay out a clear path forward.',
        }
        msg = templates.get((state, what))
        if msg: return msg
        iw = {1:'mildly',2:'somewhat',3:'quite',4:'very',5:'intensely'}.get(intensity,'')
        wp = {'now':'right now','within_15_min':'in the next few minutes',
              'later_today':'a bit later today','tonight':'this evening',
              'tomorrow_morning':'tomorrow morning'}.get(when,'soon')
        return f'You seem {iw} {state} right now. Try {what.replace("_"," ")} {wp}.'

de = DecisionEngine()
print('✅ Decision engine ready')

# Quick demo
demo = [('anxious',4,2,5,'morning'), ('focused',4,5,2,'afternoon'),
        ('tired',3,1,2,'evening'), ('overwhelmed',5,2,5,'morning')]
print('\n🧭 Decision Engine Demo:')
print(f'{"State":<12} {"Int":<5} {"Energy":<8} {"Stress":<8} {"What":<18} {"When"}')
print('-'*65)
for state, intensity, energy, stress, tod in demo:
    what = de.decide_what(state, intensity, energy, stress)
    when = de.decide_when(state, intensity, energy, stress, tod)
    print(f'{state:<12} {intensity:<5} {energy:<8} {stress:<8} {what:<18} {when}')

## 🎯 Cell 9 — Uncertainty Modeling (Part 4)

In [ ]:
class UncertaintyModule:
    CONF_THRESHOLD   = 0.45
    MARGIN_THRESHOLD = 0.15
    SHORT_WORDS      = 5

    def assess(self, confidence, proba_top2, text, energy, stress, face_hint):
        margin = proba_top2[-1] - proba_top2[-2]
        flags  = []
        if confidence < self.CONF_THRESHOLD:       flags.append('low_confidence')
        if margin     < self.MARGIN_THRESHOLD:      flags.append('low_margin')
        if len(str(text).split()) < self.SHORT_WORDS: flags.append('short_text')
        if (not pd.isna(energy) and not pd.isna(stress)
            and float(energy) >= 4 and float(stress) >= 4): flags.append('contradictory_signals')
        if str(face_hint) in ('unknown','nan',''): flags.append('missing_face_hint')
        return confidence, int(len(flags) >= 2), flags

um = UncertaintyModule()

# Get probabilities
proba_all = state_model.predict_proba(X_test)
top2_all  = np.sort(proba_all, axis=1)[:, -2:]
confs     = top2_all[:, -1]

print(f'📊 Confidence stats:')
print(f'   Mean:   {confs.mean():.3f}')
print(f'   Min:    {confs.min():.3f}')
print(f'   < 0.45: {(confs < 0.45).sum()} samples flagged as uncertain')

# Confidence histogram
plt.figure(figsize=(8,4))
plt.hist(confs, bins=20, color='steelblue', edgecolor='white')
plt.axvline(0.45, color='red', linestyle='--', label='Uncertainty threshold')
plt.xlabel('Confidence Score'); plt.ylabel('Count')
plt.title('🎯 Prediction Confidence Distribution')
plt.legend(); plt.tight_layout(); plt.show()

## 📦 Cell 10 — Generate Full predictions.csv

In [ ]:
results = []
for i, (idx, row) in enumerate(test_df.iterrows()):
    state     = y_pred_labels[i]
    intensity = int(int_pred[i])
    conf      = float(confs[i])
    top2      = top2_all[i]
    energy    = row.get('energy_level', np.nan)
    stress    = row.get('stress_level', np.nan)
    time_od   = row.get('time_of_day', 'morning')
    face      = row.get('face_emotion_hint', 'unknown')
    text      = row.get('journal_text', '')

    what = de.decide_what(state, intensity, energy, stress)
    when = de.decide_when(state, intensity, energy, stress, time_od)
    msg  = de.message(state, intensity, what, when)
    final_conf, unc_flag, unc_reasons = um.assess(conf, top2, text, energy, stress, face)

    results.append({
        'id': row['id'],
        'predicted_state': state,
        'predicted_intensity': intensity,
        'confidence': round(final_conf, 4),
        'uncertain_flag': unc_flag,
        'what_to_do': what,
        'when_to_do': when,
        'supportive_message': msg,
    })

predictions_df = pd.DataFrame(results)
predictions_df.to_csv('predictions.csv', index=False)
print(f'✅ Saved predictions.csv ({len(predictions_df)} rows)')
predictions_df.head(10)

## 🔬 Cell 11 — Ablation Study (Part 6)

In [ ]:
print('=' * 50)
print('ABLATION STUDY — Text-only vs Text + Metadata')
print('=' * 50)

le2 = LabelEncoder()
y_tr = le2.fit_transform(train_df['emotional_state'])
y_te = le2.transform(test_df_full['emotional_state'])

# TEXT ONLY
fe_text = FeatureEngineer()
Xt_tr = fe_text.text_only_fit_transform(train_df)
Xt_te = fe_text.text_only_transform(test_df_full)
clf_t = xgb.XGBClassifier(n_estimators=100, use_label_encoder=False,
                           eval_metric='mlogloss', random_state=42)
clf_t.fit(Xt_tr, y_tr)
acc_text = accuracy_score(y_te, clf_t.predict(Xt_te))

# TEXT + METADATA
fe_full = FeatureEngineer()
Xf_tr = fe_full.fit_transform(train_df)
Xf_te = fe_full.transform(test_df_full)
clf_f = xgb.XGBClassifier(n_estimators=100, use_label_encoder=False,
                           eval_metric='mlogloss', random_state=42)
clf_f.fit(Xf_tr, y_tr)
acc_full = accuracy_score(y_te, clf_f.predict(Xf_te))

print(f'🔤 Text-only Accuracy:       {acc_text:.3f}')
print(f'📊 Text + Metadata Accuracy: {acc_full:.3f}')
print(f'📈 Metadata Gain:            {acc_full - acc_text:+.3f}')

# Bar chart
plt.figure(figsize=(6,4))
bars = plt.bar(['Text Only', 'Text + Metadata'], [acc_text, acc_full],
               color=['#4a9eda','#2d7a4f'], width=0.4, edgecolor='white')
for bar, val in zip(bars, [acc_text, acc_full]):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
             f'{val:.3f}', ha='center', fontweight='bold')
plt.ylim(0, 1.1)
plt.title('Ablation Study — Accuracy Comparison')
plt.ylabel('Accuracy')
plt.tight_layout(); plt.show()

## 🔴 Cell 12 — Error Analysis (Part 7)

In [ ]:
print('ERROR ANALYSIS — 10 Failure Cases')
print('=' * 55)

failures = []
for i, (idx, row) in enumerate(test_df_full.iterrows()):
    true_s = row['emotional_state']
    pred_s = y_pred_labels[i]
    true_i = row['intensity']
    pred_i = int_pred[i]
    if true_s != pred_s or abs(true_i - pred_i) >= 2:
        failures.append({
            'id': row['id'], 'text': row['journal_text'],
            'true_state': true_s, 'pred_state': pred_s,
            'true_int': true_i, 'pred_int': pred_i,
            'confidence': round(float(confs[i]),3),
            'margin': round(float(top2_all[i,-1]-top2_all[i,-2]),3),
            'sleep': row['sleep_hours'], 'energy': row['energy_level'],
            'stress': row['stress_level'], 'face': row['face_emotion_hint'],
        })

print(f'Total failures: {len(failures)}')
for k, f in enumerate(failures[:10]):
    print(f'\n--- Case {k+1} | ID {f["id"]} ---')
    print(f'  Text:      "{f["text"]}"')
    print(f'  True:      {f["true_state"]} (intensity {f["true_int"]})')
    print(f'  Predicted: {f["pred_state"]} (intensity {f["pred_int"]})')
    print(f'  Conf: {f["confidence"]}  Margin: {f["margin"]}')
    print(f'  sleep={f["sleep"]}, energy={f["energy"]}, stress={f["stress"]}, face={f["face"]}')
    # Diagnose
    if len(str(f['text']).split()) < 5:
        print('  ⚠ SHORT TEXT — insufficient signal')
    if f['confidence'] < 0.45:
        print('  ⚠ LOW CONFIDENCE — model uncertain')
    if f['margin'] < 0.15:
        print('  ⚠ LOW MARGIN — two classes nearly tied')
    if f['true_state'] in ('calm','content','hopeful') and f['pred_state'] in ('calm','content','hopeful'):
        print('  ⚠ POSITIVE STATE CONFUSION — calm/content/hopeful share vocabulary')
    if {f['true_state'],f['pred_state']} == {'sad','tired'}:
        print('  ⚠ SAD vs TIRED — low-energy language is ambiguous')

## 🛡️ Cell 13 — Robustness Tests (Part 9)

In [ ]:
print('ROBUSTNESS TESTS')
print('='*55)

edge_cases = pd.DataFrame([
    {'id':9001,'journal_text':'ok','ambience_type':'forest','duration_min':10,
     'sleep_hours':np.nan,'energy_level':np.nan,'stress_level':5,
     'time_of_day':'morning','previous_day_mood':'neutral',
     'face_emotion_hint':'unknown','reflection_quality':'low'},
    {'id':9002,'journal_text':'fine.','ambience_type':'ocean','duration_min':15,
     'sleep_hours':8.0,'energy_level':1,'stress_level':1,
     'time_of_day':'morning','previous_day_mood':'excellent',
     'face_emotion_hint':'happy','reflection_quality':'low'},
    {'id':9003,'journal_text':'I feel amazing but also terrified and exhausted all at once',
     'ambience_type':'mountain','duration_min':30,
     'sleep_hours':4.0,'energy_level':5,'stress_level':5,
     'time_of_day':'afternoon','previous_day_mood':'terrible',
     'face_emotion_hint':'tense','reflection_quality':'medium'},
])

X_edge   = fe.transform(edge_cases)
e_preds  = le.inverse_transform(state_model.predict(X_edge))
e_proba  = state_model.predict_proba(X_edge)
e_top2   = np.sort(e_proba, axis=1)[:, -2:]
e_confs  = e_top2[:, -1]
e_ints   = np.clip(np.round(intensity_model.predict(X_edge)), 1, 5).astype(int)

for i, row in edge_cases.iterrows():
    state     = e_preds[i]
    intensity = int(e_ints[i])
    conf, unc_flag, reasons = um.assess(
        float(e_confs[i]), e_top2[i],
        row.journal_text, row.energy_level, row.stress_level, row.face_emotion_hint
    )
    what = de.decide_what(state, intensity, row.energy_level, row.stress_level)
    when = de.decide_when(state, intensity, row.energy_level, row.stress_level, row.time_of_day)
    print(f'\nID {row.id}: "{row.journal_text}"')
    print(f'  → State: {state} | Intensity: {intensity} | Conf: {conf:.3f} | Uncertain: {unc_flag}')
    print(f'  → What: {what} | When: {when}')
    print(f'  → Flags: {reasons}')

## ⬇️ Cell 14 — Download predictions.csv

In [ ]:
from google.colab import files
files.download('predictions.csv')
print('✅ predictions.csv downloaded!')

---
## ✅ All Parts Complete

| Part | Status |
|------|--------|
| 1 — Emotional State Prediction | ✅ |
| 2 — Intensity Prediction | ✅ |
| 3 — Decision Engine (What + When) | ✅ |
| 4 — Uncertainty Modeling | ✅ |
| 5 — Feature Understanding | ✅ |
| 6 — Ablation Study | ✅ |
| 7 — Error Analysis | ✅ |
| 8 — Edge Plan | See EDGE_PLAN.md |
| 9 — Robustness Tests | ✅ |
| Bonus — Supportive Messages | ✅ |